# Two-Stage Model for Vt Deviation Prediction

## Approach

This notebook implements a **two-stage modeling approach** to predict Vt deviations:

1. **Stage 1 (Classification)**: Identify wafers with significant deviations (|deviation| > 0.02)
   - Uses CatBoost classifier
   - Outputs probability of significant deviation
   - Evaluated on AUC-ROC and precision/recall

2. **Stage 2 (Regression)**: Predict exact magnitude of deviation
   - Uses CatBoost regressor with sample weighting
   - Weights samples by classifier confidence
   - High-confidence predictions get more weight

## Benefits

- **Better focus**: Classifier identifies problematic wafers first
- **Adaptive weighting**: Regression focuses on high-confidence regions
- **Interpretability**: Two clear objectives (classification + regression)

In [12]:
import pandas as pd
import numpy as np
from pathlib import Path
from catboost import CatBoostRegressor, CatBoostClassifier, Pool
from sklearn.metrics import (
    mean_squared_error, r2_score, mean_absolute_error, 
    roc_auc_score, precision_recall_curve, auc,
    confusion_matrix, classification_report
)
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from tqdm.auto import tqdm

sns.set_style('whitegrid')
pd.set_option('display.max_columns', 50)

print(f"Started: {datetime.now()}")

Started: 2026-03-07 16:57:01.549870


## 1. Load Data and Ideal Baseline

In [13]:
# Detect project root
root = Path.cwd()
while not (root / 'CLAUDE.md').exists() and root != root.parent:
    root = root.parent

print(f"Project root: {root}")

# Load feature data
train_df = pd.read_parquet(root / 'outputs/features/train.parquet')
val_df = pd.read_parquet(root / 'outputs/features/val.parquet')

# Load Vt data
vt_raw = pd.read_csv(root / 'data/response_updated.csv')
vt_df = vt_raw.groupby('WAFER_SCRIBE').agg({
    'VALUE': 'mean',
    'LOT_ID': 'first',
    'PARAM_END_DATETIME': 'first'
}).reset_index()
vt_df.columns = ['WAFER_SCRIBE', 'vt_actual', 'LOT_ID', 'PARAM_END_DATETIME']
vt_df = vt_df.drop_duplicates(subset='WAFER_SCRIBE', keep='first')

# Compute deviation from ideal
IDEAL_VT = 0.5
vt_df['vt_deviation_from_ideal'] = vt_df['vt_actual'] - IDEAL_VT
vt_df['vt_abs_deviation'] = np.abs(vt_df['vt_deviation_from_ideal'])

# Merge with features
train_df = train_df.merge(vt_df[['WAFER_SCRIBE', 'vt_actual', 'vt_deviation_from_ideal', 'vt_abs_deviation']], 
                          on='WAFER_SCRIBE', how='left')
val_df = val_df.merge(vt_df[['WAFER_SCRIBE', 'vt_actual', 'vt_deviation_from_ideal', 'vt_abs_deviation']], 
                      on='WAFER_SCRIBE', how='left')

print(f"Train shape: {train_df.shape}")
print(f"Val shape: {val_df.shape}")

Project root: d:\capstone_pipeline
Train shape: (13510, 10309)
Val shape: (3307, 10309)


In [14]:
# Load nominal baseline (from notebook 13)
nominal_baseline = pd.read_csv(root / 'outputs/features/nominal_sensor_baseline.csv',
                               index_col=0).squeeze()

print(f"Loaded nominal baseline for {len(nominal_baseline)} sensors")

Loaded nominal baseline for 10249 sensors


## 2. Add Deviation Features

Compute deviation from the ideal baseline for each sensor.

In [15]:
# Identify sensor columns
exclude_cols = {'WAFER_SCRIBE', 'LOT_ID', 'PARAM_END_DATETIME', 'is_outlier', 
                'vt_actual', 'vt_deviation_from_ideal', 'vt_abs_deviation'}
all_cols = [c for c in train_df.columns if c not in exclude_cols]
sensor_cols = [c for c in all_cols if c in nominal_baseline.index]

print(f"Total features: {len(all_cols)}")
print(f"Sensor features with nominal baseline: {len(sensor_cols)}")

Total features: 10302
Sensor features with nominal baseline: 10249


In [16]:
def add_deviation_features(df, nominal_baseline, sensor_cols):
    """Add deviation-from-nominal features"""
    new_cols = {}
    
    for col in tqdm(sensor_cols, desc="Adding deviation features"):
        new_cols[f'{col}__dev_from_nominal'] = np.abs(df[col] - nominal_baseline[col])
        
        if nominal_baseline[col] != 0:
            new_cols[f'{col}__rel_dev'] = (df[col] - nominal_baseline[col]) / np.abs(nominal_baseline[col])
    
    return pd.concat([df, pd.DataFrame(new_cols, index=df.index)], axis=1)

# Apply the function to create deviation features
print("Adding deviation features to training set...")
train_df_dev = add_deviation_features(train_df, nominal_baseline, sensor_cols)

print("Adding deviation features to validation set...")
val_df_dev = add_deviation_features(val_df, nominal_baseline, sensor_cols)

print(f"\nTrain shape with deviation features: {train_df_dev.shape}")
print(f"Val shape with deviation features: {val_df_dev.shape}")

Adding deviation features to training set...


Adding deviation features: 100%|██████████| 10249/10249 [00:02<00:00, 3613.52it/s]


Adding deviation features to validation set...


Adding deviation features: 100%|██████████| 10249/10249 [00:01<00:00, 5595.75it/s]



Train shape with deviation features: (13510, 27737)
Val shape with deviation features: (3307, 27737)


In [17]:
# Add summary deviation metrics
dev_cols = [c for c in train_df_dev.columns if '__dev_from_nominal' in c]
rel_dev_cols = [c for c in train_df_dev.columns if '__rel_dev' in c]

train_df_dev['total_abs_deviation'] = train_df_dev[dev_cols].sum(axis=1)
train_df_dev['mean_abs_deviation'] = train_df_dev[dev_cols].mean(axis=1)
train_df_dev['max_abs_deviation'] = train_df_dev[dev_cols].max(axis=1)
train_df_dev['mean_rel_deviation'] = train_df_dev[rel_dev_cols].mean(axis=1)

val_df_dev['total_abs_deviation'] = val_df_dev[dev_cols].sum(axis=1)
val_df_dev['mean_abs_deviation'] = val_df_dev[dev_cols].mean(axis=1)
val_df_dev['max_abs_deviation'] = val_df_dev[dev_cols].max(axis=1)
val_df_dev['mean_rel_deviation'] = val_df_dev[rel_dev_cols].mean(axis=1)

print(f"Added 4 summary deviation metrics")

Added 4 summary deviation metrics


## 3. Prepare Feature Matrix

In [18]:
# Prepare feature columns
feature_cols = [c for c in train_df_dev.columns if c not in exclude_cols]

X_train = train_df_dev[feature_cols].copy()
y_train = train_df_dev['vt_deviation_from_ideal'].copy()

X_val = val_df_dev[feature_cols].copy()
y_val = val_df_dev['vt_deviation_from_ideal'].copy()

# Identify categorical features
cat_features = [col for col in feature_cols if 
                col == 'LOT_ID' or '__EQUIP' in col or '__POSITION' in col or 
                col.startswith('first_step') or col.startswith('last_step')]

# Convert to string and fill NaN values (critical for CatBoost)
for col in cat_features:
    if col in X_train.columns:
        X_train[col] = X_train[col].fillna('missing').astype(str)
        X_val[col] = X_val[col].fillna('missing').astype(str)

print(f"Features: {len(feature_cols)}")
print(f"Categorical features: {len(cat_features)}")
print(f"Train samples: {len(X_train)}")
print(f"Val samples: {len(X_val)}")

Features: 27734
Categorical features: 48
Train samples: 13510
Val samples: 3307


In [19]:
# Load baseline model from notebook 13 (or train if not available)
model_path = root / 'outputs/models/catboost_baseline_ideal.cbm'

if model_path.exists():
    print(f"Loading saved baseline model from {model_path.name}...")
    model_baseline = CatBoostRegressor()
    model_baseline.load_model(str(model_path))
    y_pred_baseline = model_baseline.predict(X_val)
    print("✓ Baseline model loaded successfully")
else:
    print(f"Model file not found. Training baseline model...")
    train_pool = Pool(X_train, y_train, cat_features=cat_features)
    val_pool = Pool(X_val, y_val, cat_features=cat_features)

    model_baseline = CatBoostRegressor(
        iterations=600,
        learning_rate=0.05,
        depth=6,
        eval_metric='RMSE',
        early_stopping_rounds=50,
        random_seed=42,
        verbose=100,
        task_type='GPU' if CatBoostRegressor().is_gpu_available() else 'CPU'
    )

    model_baseline.fit(train_pool, eval_set=val_pool)
    y_pred_baseline = model_baseline.predict(X_val)
    
    model_baseline.save_model(str(model_path))
    print(f"✓ Model saved to {model_path.name}")

# Evaluate baseline
r2_baseline = r2_score(y_val, y_pred_baseline)
rmse_baseline = np.sqrt(mean_squared_error(y_val, y_pred_baseline))
mae_baseline = mean_absolute_error(y_val, y_pred_baseline)

print(f"\nBaseline Performance:")
print(f"  R²: {r2_baseline:.4f}")
print(f"  RMSE: {rmse_baseline:.4f}")
print(f"  MAE: {mae_baseline:.4f}")

Loading saved baseline model from catboost_baseline_ideal.cbm...
✓ Baseline model loaded successfully

Baseline Performance:
  R²: 0.3978
  RMSE: 0.0927
  MAE: 0.0676


In [20]:
# Define threshold for "significant deviation"
SIGNIFICANT_DEV_THRESHOLD = 0.02

y_train_class = (train_df_dev['vt_abs_deviation'] > SIGNIFICANT_DEV_THRESHOLD).astype(int)
y_val_class = (val_df_dev['vt_abs_deviation'] > SIGNIFICANT_DEV_THRESHOLD).astype(int)

print(f"Classification target distribution:")
print(f"  Train: {y_train_class.sum():,} / {len(y_train_class):,} ({y_train_class.mean():.2%}) significant deviations")
print(f"  Val: {y_val_class.sum():,} / {len(y_val_class):,} ({y_val_class.mean():.2%}) significant deviations")

Classification target distribution:
  Train: 11,647 / 13,510 (86.21%) significant deviations
  Val: 2,851 / 3,307 (86.21%) significant deviations


### Stage 2: Weighted Regression

Use classifier confidence to weight regression samples.

In [ ]:
# Stage 2: Use classifier confidence to weight regression
# High-confidence predictions get more weight in regression

train_pred_prob = model_classifier.predict_proba(X_train)[:, 1]
regression_weights = np.abs(train_pred_prob - 0.5) * 2  # Scale 0.5-1.0 → 0-1

print(f"Regression weights statistics:")
print(f"  Mean: {regression_weights.mean():.4f}")
print(f"  Std: {regression_weights.std():.4f}")
print(f"  Range: [{regression_weights.min():.4f}, {regression_weights.max():.4f}]")

train_pool_twostage = Pool(X_train, y_train, cat_features=cat_features, weight=regression_weights)
val_pool = Pool(X_val, y_val, cat_features=cat_features)

model_twostage = CatBoostRegressor(
    iterations=500,
    learning_rate=0.1,
    depth=6,
    eval_metric='RMSE',
    early_stopping_rounds=30,
    random_seed=42,
    verbose=100,
    task_type='CPU'
)

print("\n=== Training Stage 2: Weighted Regression ===")
model_twostage.fit(train_pool_twostage, eval_set=val_pool)

# Evaluate
y_pred_twostage = model_twostage.predict(X_val)
r2_twostage = r2_score(y_val, y_pred_twostage)
rmse_twostage = np.sqrt(mean_squared_error(y_val, y_pred_twostage))
mae_twostage = mean_absolute_error(y_val, y_pred_twostage)

print(f"\nTwo-Stage Model Performance:")
print(f"  R²: {r2_twostage:.4f} ")
print(f"  RMSE: {rmse_twostage:.4f} ")
print(f"  MAE: {mae_twostage:.4f} ")

Regression weights statistics:
  Mean: 0.7280
  Std: 0.0978
  Range: [0.0002, 0.9431]

=== Training Stage 2: Weighted Regression ===
0:	learn: 0.1160956	test: 0.1171817	best: 0.1171817 (0)	total: 1.26s	remaining: 10m 29s


## 6. Comprehensive Performance Visualization

In [ ]:
# Stage 2: Use classifier confidence to weight regression
# High-confidence predictions get more weight in regression

train_pred_prob = model_classifier.predict_proba(X_train)[:, 1]
regression_weights = np.abs(train_pred_prob - 0.5) * 2  # Scale 0.5-1.0 → 0-1

print(f"Regression weights statistics:")
print(f"  Mean: {regression_weights.mean():.4f}")
print(f"  Std: {regression_weights.std():.4f}")
print(f"  Range: [{regression_weights.min():.4f}, {regression_weights.max():.4f}]")

train_pool_twostage = Pool(X_train, y_train, cat_features=cat_features, weight=regression_weights)
val_pool = Pool(X_val, y_val, cat_features=cat_features)

model_twostage = CatBoostRegressor(
    iterations=500,
    learning_rate=0.1,
    depth=6,
    eval_metric='RMSE',
    early_stopping_rounds=30,
    random_seed=42,
    verbose=100,
    task_type='CPU'
)

print("\n=== Training Stage 2: Weighted Regression ===")
model_twostage.fit(train_pool_twostage, eval_set=val_pool)

# Evaluate
y_pred_twostage = model_twostage.predict(X_val)
r2_twostage = r2_score(y_val, y_pred_twostage)
rmse_twostage = np.sqrt(mean_squared_error(y_val, y_pred_twostage))
mae_twostage = mean_absolute_error(y_val, y_pred_twostage)

print(f"\nTwo-Stage Model Performance:")
print(f"  R²: {r2_twostage:.4f}")
print(f"  RMSE: {rmse_twostage:.4f}")
print(f"  MAE: {mae_twostage:.4f}")

## 5. Comprehensive Performance Visualization

In [ ]:
# Compute residuals
residuals_twostage = y_val - y_pred_twostage

# Create comprehensive visualization
fig = plt.figure(figsize=(16, 10))
gs = fig.add_gridspec(2, 3, hspace=0.3, wspace=0.3)

# 1. Predictions vs Actual (large plot)
ax1 = fig.add_subplot(gs[0:2, 0:2])
scatter = ax1.scatter(y_val, y_pred_twostage, alpha=0.4, s=30, c=np.abs(residuals_twostage), 
                      cmap='RdYlGn_r', edgecolors='black', linewidth=0.3)
ax1.plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], 
         'r--', linewidth=2, label='Perfect Prediction')

# Add confidence bands (±RMSE)
x_line = np.linspace(y_val.min(), y_val.max(), 100)
ax1.fill_between(x_line, x_line - rmse_twostage, x_line + rmse_twostage, 
                 alpha=0.2, color='blue', label=f'±1 RMSE ({rmse_twostage:.4f})')

ax1.set_xlabel('Actual Vt Deviation from Ideal', fontsize=13, fontweight='bold')
ax1.set_ylabel('Predicted Vt Deviation from Ideal', fontsize=13, fontweight='bold')
ax1.set_title('Two-Stage Model: Predictions vs Actual', fontsize=15, fontweight='bold')
ax1.legend(fontsize=11, loc='upper left')
ax1.grid(True, alpha=0.3)

# Add colorbar
cbar = plt.colorbar(scatter, ax=ax1)
cbar.set_label('Absolute Error', fontsize=11)

# Add metrics text box
textstr = f'R² = {r2_twostage:.4f}\\nRMSE = {rmse_twostage:.4f}\\nMAE = {mae_twostage:.4f}'
props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
ax1.text(0.05, 0.95, textstr, transform=ax1.transAxes, fontsize=12,
         verticalalignment='top', bbox=props)

# 2. Residuals vs Predicted
ax2 = fig.add_subplot(gs[0, 2])
ax2.scatter(y_pred_twostage, residuals_twostage, alpha=0.4, s=20, 
            edgecolors='black', linewidth=0.3)
ax2.axhline(y=0, color='r', linestyle='--', linewidth=2)
ax2.axhline(y=rmse_twostage, color='orange', linestyle=':', linewidth=1.5, label='+RMSE')
ax2.axhline(y=-rmse_twostage, color='orange', linestyle=':', linewidth=1.5, label='-RMSE')
ax2.set_xlabel('Predicted Value', fontsize=11)
ax2.set_ylabel('Residual (Actual - Pred)', fontsize=11)
ax2.set_title('Residual Plot', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.legend(fontsize=9)

# 3. Error Distribution
ax3 = fig.add_subplot(gs[1, 2])
ax3.hist(residuals_twostage, bins=50, alpha=0.7, edgecolor='black', color='steelblue')
ax3.axvline(x=0, color='red', linestyle='--', linewidth=2, label='Zero Error')
ax3.axvline(x=residuals_twostage.mean(), color='orange', linestyle='-', linewidth=2, 
            label=f'Mean Error ({residuals_twostage.mean():.4f})')
ax3.set_xlabel('Residual', fontsize=11)
ax3.set_ylabel('Frequency', fontsize=11)
ax3.set_title('Error Distribution', fontsize=12, fontweight='bold')
ax3.legend(fontsize=9)
ax3.grid(True, alpha=0.3, axis='y')

plt.suptitle('Two-Stage Model Performance Analysis', 
             fontsize=16, fontweight='bold', y=0.995)

plt.savefig(root / 'outputs/plots/twostage_model_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Saved: outputs/plots/twostage_model_analysis.png")

## 6. Detailed Performance Summary

In [ ]:
# Print detailed performance summary table
print("\n" + "="*80)
print("TWO-STAGE MODEL DETAILED PERFORMANCE SUMMARY")
print("="*80)

# Compute additional statistics
abs_errors_twostage = np.abs(residuals_twostage)

summary_data = {
    'Metric': [
        'R² Score',
        'RMSE',
        'MAE',
        'Median Absolute Error',
        'Mean Error (Bias)',
        'Std Dev of Errors',
        '90th Percentile Error',
        '95th Percentile Error',
        '99th Percentile Error',
        'Max Error',
        '% Predictions within ±0.02',
        '% Predictions within ±0.05',
        '% Predictions within ±0.10'
    ],
    'Value': [
        f'{r2_twostage:.4f}',
        f'{rmse_twostage:.4f}',
        f'{mae_twostage:.4f}',
        f'{np.median(abs_errors_twostage):.4f}',
        f'{residuals_twostage.mean():.4f}',
        f'{residuals_twostage.std():.4f}',
        f'{np.percentile(abs_errors_twostage, 90):.4f}',
        f'{np.percentile(abs_errors_twostage, 95):.4f}',
        f'{np.percentile(abs_errors_twostage, 99):.4f}',
        f'{abs_errors_twostage.max():.4f}',
        f'{(abs_errors_twostage <= 0.02).mean() * 100:.2f}%',
        f'{(abs_errors_twostage <= 0.05).mean() * 100:.2f}%',
        f'{(abs_errors_twostage <= 0.10).mean() * 100:.2f}%'
    ]
}

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))

print("\n" + "="*80)
print("KEY INSIGHTS:")
print("="*80)
print(f"• Two-Stage Model R²: {r2_twostage:.4f}")
print(f"• Prediction Error (RMSE): {rmse_twostage:.4f}")
print(f"• Mean Absolute Error: {mae_twostage:.4f}")
print(f"\n• Stage 1 Classifier identified {y_val_class.sum():,} wafers with significant deviations")
print(f"• Classifier AUC-ROC: {auc_score:.4f}")
print(f"• Classifier AUC-PR: {pr_auc:.4f}")
print(f"\n• {(abs_errors_twostage <= 0.05).sum():,} out of {len(y_val):,} predictions within ±0.05 Vt")
print(f"• {(abs_errors_twostage <= 0.10).sum():,} out of {len(y_val):,} predictions within ±0.10 Vt")
print(f"\n• Median prediction error: {np.median(abs_errors_twostage):.4f}")
print(f"• 95% of predictions have error ≤ {np.percentile(abs_errors_twostage, 95):.4f}")
print("="*80)

## 7. Stage 1 Classifier Detailed Analysis

In [ ]:
# Save two-stage models
classifier_path = root / 'outputs/models/twostage_classifier.cbm'
regressor_path = root / 'outputs/models/twostage_regressor.cbm'

model_classifier.save_model(str(classifier_path))
model_twostage.save_model(str(regressor_path))

print(f"✓ Saved Stage 1 Classifier to: {classifier_path.name}")
print(f"✓ Saved Stage 2 Regressor to: {regressor_path.name}")

# Save performance metrics
metrics_summary = {
    'model': 'Two-Stage',
    'r2': r2_twostage,
    'rmse': rmse_twostage,
    'mae': mae_twostage,
    'classifier_auc': auc_score,
    'classifier_pr_auc': pr_auc,
    'threshold': SIGNIFICANT_DEV_THRESHOLD,
    'trained_at': str(datetime.now())
}

import json
with open(root / 'outputs/twostage_model_metrics.json', 'w') as f:
    json.dump(metrics_summary, f, indent=2)

print("✓ Saved metrics to: twostage_model_metrics.json")

In [ ]:
## 8. Save Models